In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import psycopg2

In [2]:
conn = psycopg2.connect(
    host="localhost",
    port="5432",
    database="ITSM_Project",
    user="postgres",
    password="SYSTEM"
)

In [3]:
print("PostgreSQL connection successful!")

PostgreSQL connection successful!


In [4]:
## Load Data from PostgreSQL
query = """
SELECT *
FROM itsm_ops.ticket;
"""

df = pd.read_sql(query, conn)

print("Data loaded successfully!")
print("Shape:", df.shape)

C:\Users\Medhya Khatri\AppData\Local\Temp\ipykernel_27128\206125525.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Data loaded successfully!
Shape: (180000, 20)


In [5]:
df.head()

,ticket_id,ticket_number,company_id,requester_user_id,assigned_agent_id,opened_at,resolved_at,closed_at,ticket_type,category,sub_category,priority,impact,urgency,ticket_status,source_channel,resolution_code,reopen_count,customer_satisfaction_rating,created_month
0,1,INC-2021-00000001,123,1197,807,2021-06-27 17:01:00,2021-07-01 19:14:00,2021-07-02 04:08:00,Incident,Software,Patch Update,P3,Medium,Medium,Closed,Email,User Educated,0,1.0,2021-06-01
1,2,INC-2021-00000002,249,1091,904,2021-06-27 17:01:00,2021-07-01 19:14:00,2021-07-03 18:38:00,Incident,Software,Patch Update,P3,Medium,Medium,Closed,Email,Known Error Workaround,0,5.0,2021-06-01
2,3,INC-2021-00000003,176,343,127,2021-06-27 17:01:00,2021-07-01 19:14:00,2021-07-03 10:32:00,Incident,Software,Patch Update,P3,Medium,Medium,Closed,Email,User Educated,0,NaN,2021-06-01
3,5,INC-2021-00000005,288,237,477,2021-06-27 17:01:00,2021-07-01 19:14:00,2021-07-03 19:23:00,Incident,Software,Patch Update,P3,Medium,Medium,Closed,Email,Patch Applied,0,1.0,2021-06-01
4,7,INC-2021-00000007,235,1146,773,2021-06-27 17:01:00,2021-07-01 19:14:00,2021-07-02 00:25:00,Incident,Software,Patch Update,P3,Medium,Medium,Closed,Email,Password Reset Completed,0,1.0,2021-06-01


In [6]:
## Load Data from PostgreSQL
sla_query = """
SELECT *
FROM itsm_ops.ticket_sla_log;
"""

sla_df = pd.read_sql(sla_query, conn)

print("SLA data loaded successfully!")
print(sla_df.shape)

C:\Users\Medhya Khatri\AppData\Local\Temp\ipykernel_27128\4210799565.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sla_df = pd.read_sql(sla_query, conn)


SLA data loaded successfully!
(360000, 8)


In [7]:
## Merge Ticket and SLA Data

merged_df = pd.merge(
    df,
    sla_df,
    on="ticket_id",
    how="left"
)

print("DataFrames merged successfully!")
print("Merged Shape:", merged_df.shape)

DataFrames merged successfully!
Merged Shape: (360000, 27)


In [8]:
## Merged DataFrame

print("Shape of merged DataFrame:", merged_df.shape)

print("\nData Types:")
print(merged_df.dtypes)

print("\nMissing Values:")
print(merged_df.isnull().sum())

print("\nDuplicate Rows:", merged_df.duplicated().sum())

print("\nDuplicate Ticket IDs:", merged_df["ticket_id"].duplicated().sum())

Shape of merged DataFrame: (360000, 27)

Data Types:
ticket_id                                int64
ticket_number                           object
company_id                               int64
requester_user_id                        int64
assigned_agent_id                        int64
opened_at                       datetime64[ns]
resolved_at                     datetime64[ns]
closed_at                       datetime64[ns]
ticket_type                             object
category                                object
sub_category                            object
priority                                object
impact                                  object
urgency                                 object
ticket_status                           object
source_channel                          object
resolution_code                         object
reopen_count                             int64
customer_satisfaction_rating           float64
created_month                           object
sla_log

In [9]:
##Analyze Missing Customer Satisfaction Ratings

print("Missing CSAT:", merged_df["customer_satisfaction_rating"].isnull().sum())

print("\nCSAT values available:")
print(merged_df["customer_satisfaction_rating"].value_counts(dropna=False).sort_index())

Missing CSAT: 61446

CSAT values available:
customer_satisfaction_rating
1.0    59022
2.0    59496
3.0    59758
4.0    60446
5.0    59832
NaN    61446
Name: count, dtype: int64


In [10]:
##Check Data Quality

print("Negative SLA actual minutes:",
      (merged_df["actual_minutes"] < 0).sum())

print("Negative SLA target minutes:",
      (merged_df["sla_target_minutes"] < 0).sum())

print("Negative breach minutes:",
      (merged_df["breach_minutes"] < 0).sum())

print("Invalid CSAT ratings:",
      (~merged_df["customer_satisfaction_rating"].isin([1, 2, 3, 4, 5]) &
       merged_df["customer_satisfaction_rating"].notna()).sum())

Negative SLA actual minutes: 0
Negative SLA target minutes: 0
Negative breach minutes: 0
Invalid CSAT ratings: 0


In [11]:
##Create Cleaned Working DataFrame

cleaned_df = merged_df.copy()

print("Cleaned DataFrame created successfully!")
print("Shape:", cleaned_df.shape)

Cleaned DataFrame created successfully!
Shape: (360000, 27)


In [12]:
##Check Categorical Values

categorical_columns = cleaned_df.select_dtypes(include="object").columns

for col in categorical_columns:
    print(f"\n{col}:")
    print(cleaned_df[col].value_counts(dropna=False))


ticket_number:
ticket_number
INC-2021-00000001    2
INC-2021-00151511    2
INC-2021-00151501    2
INC-2021-00151503    2
INC-2021-00151504    2
                    ..
INC-2021-00075613    2
INC-2021-00075614    2
INC-2021-00075615    2
INC-2021-00075616    2
INC-2021-00179995    2
Name: count, Length: 180000, dtype: int64

ticket_type:
ticket_type
Incident    360000
Name: count, dtype: int64

category:
category
Software    360000
Name: count, dtype: int64

sub_category:
sub_category
Patch Update    360000
Name: count, dtype: int64

priority:
priority
P3    360000
Name: count, dtype: int64

impact:
impact
Medium    360000
Name: count, dtype: int64

urgency:
urgency
Medium    360000
Name: count, dtype: int64

ticket_status:
ticket_status
Closed    360000
Name: count, dtype: int64

source_channel:
source_channel
Email    360000
Name: count, dtype: int64

resolution_code:
resolution_code
User Educated               45196
Known Error Workaround      45170
Patch Applied               45168


In [13]:
numerical_columns = cleaned_df.select_dtypes(include="number").columns

print("Numerical Columns:")
print(list(numerical_columns))

Numerical Columns:
['ticket_id', 'company_id', 'requester_user_id', 'assigned_agent_id', 'reopen_count', 'customer_satisfaction_rating', 'sla_log_id', 'sla_target_minutes', 'actual_minutes', 'breach_minutes']


In [14]:
##Check Numerical Outliers

for col in numerical_columns:
    Q1 = cleaned_df[col].quantile(0.25)
    Q3 = cleaned_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = ((cleaned_df[col] < lower_bound) |
                (cleaned_df[col] > upper_bound)).sum()

    print(f"{col}: {outliers} potential outliers")

ticket_id: 0 potential outliers
company_id: 0 potential outliers
requester_user_id: 0 potential outliers
assigned_agent_id: 0 potential outliers
reopen_count: 0 potential outliers
customer_satisfaction_rating: 0 potential outliers
sla_log_id: 0 potential outliers
sla_target_minutes: 0 potential outliers
actual_minutes: 0 potential outliers
breach_minutes: 0 potential outliers


In [15]:
cleaned_df["sla_variance_minutes"] = (
    cleaned_df["actual_minutes"] - cleaned_df["sla_target_minutes"]
)

In [21]:
print(cleaned_df[
    [
        
        "sla_variance_minutes",
    ]
].head(10))

   sla_variance_minutes
0                   226
1                  4453
2                   273
3                  4453
4                    55
5                  4453
6                   137
7                  4453
8                   270
9                  4453


In [17]:
cleaned_df["sla_performance_percentage"] = (
    cleaned_df["actual_minutes"] /
    cleaned_df["sla_target_minutes"]
) * 100

In [18]:
print(cleaned_df[
    ["sla_target_minutes", "actual_minutes", "sla_performance_percentage"]
].head(10))

   sla_target_minutes  actual_minutes  sla_performance_percentage
0                 120             346                  288.333333
1                1440            5893                  409.236111
2                 120             393                  327.500000
3                1440            5893                  409.236111
4                 120             175                  145.833333
5                1440            5893                  409.236111
6                 120             257                  214.166667
7                1440            5893                  409.236111
8                 120             390                  325.000000
9                1440            5893                  409.236111


In [19]:
cleaned_df.to_csv("cleaned_itsm_data.csv", index=False)

print("Cleaned dataset saved successfully!")
print("Shape:", cleaned_df.shape)
print("Columns:", len(cleaned_df.columns))

Cleaned dataset saved successfully!
Shape: (360000, 29)
Columns: 29


In [20]:
##Final Data Validation

print("Final DataFrame Shape:", cleaned_df.shape)
print("Total Missing Values:", cleaned_df.isnull().sum().sum())
print("Total Duplicate Rows:", cleaned_df.duplicated().sum())

print("\nData Cleaning & Preparation completed successfully!")

Final DataFrame Shape: (360000, 29)
Total Missing Values: 61446
Total Duplicate Rows: 0

Data Cleaning & Preparation completed successfully!
